In [4]:
import re, csv, os, glob
import pandas as pd



In [5]:
# ── FUNCIÓN LECTORA ──────────────────────────────────────────────────────────
def leer_csv_capology(archivo):
    """
    Los CSVs de Capology tienen doble-quoting anidado:
      - Cada fila entera entre comillas externas
      - Celdas con comas (dinero, col Adj) escapadas con ""dobles""
    No usar split(',') manual — rompe los valores monetarios.
    """
    filas = []
    with open(archivo, 'r', encoding='utf-8-sig', errors='ignore', newline='') as f:
        for linea in f:
            linea = linea.rstrip('\r\n').strip()
            if not linea:
                continue
            # 1. Quitar comilla externa de la fila completa
            if linea.startswith('"') and linea.endswith('"'):
                linea = linea[1:-1]
            # 2. ""valor"" → "valor"  (convierte al escape estándar CSV)
            linea = re.sub(r'""([^"]*?)""', r'"\1"', linea)
            # 3. Parsear — csv.reader respeta las comas dentro de campos quoted
            parsed = next(csv.reader([linea]))
            filas.append(parsed)
    return filas


In [6]:
# ── LOOP PRINCIPAL ───────────────────────────────────────────────────────────
ruta_salarios = r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Salarios'
ligas = ['La_liga', 'Premier', 'Bundesliga', 'Serie_a', 'Ligue']
acumulado = []

for liga in ligas:
    ruta_liga = os.path.join(ruta_salarios, liga)
    for archivo in glob.glob(os.path.join(ruta_liga, "*.csv")):
        try:
            filas = leer_csv_capology(archivo)
            df = pd.DataFrame(filas[1:], columns=filas[0])

            # Índices verificados contra el header real de Capology:
            # [0]Player [1]vacío [2]Weekly [3]Annual [4]Bonus [5]Total [6]Adj [7]Status [8]Pos [9]Age [10]Country [11]Club
            indices = [0, 2, 3, 4, 5, 6, 8, 9, 10, 11]
            df = df.iloc[:, indices].copy()
            df.columns = [
                'Player', 'Net_Fixed_PW_USD', 'Net_Fixed_PY_USD',
                'Net_Bonus_PY_USD', 'Net_Total_PY_USD', 'Adj_Net_Total_PY_2026',
                'Position', 'Age', 'Country', 'Club'
            ]

            # Limpiar dinero: quitar "$ " y comas de miles → float
            cols_dinero = ['Net_Fixed_PW_USD', 'Net_Fixed_PY_USD', 'Net_Bonus_PY_USD',
                           'Net_Total_PY_USD', 'Adj_Net_Total_PY_2026']
            for col in cols_dinero:
                df[col] = df[col].str.replace(r'[^\d.]', '', regex=True)
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

            df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

            # Temporada desde nombre de archivo (ej: "bundesliga_22_23" → 2022-2023)
            nombre_arc = os.path.basename(archivo).lower()
            nums = re.findall(r'\d+', nombre_arc)
            anio = nums[0] if nums else "23"
            df['Temporada'] = f"20{anio}-20{int(anio)+1}" if len(anio) == 2 else f"{anio}-{int(anio)+1}"
            df['Liga'] = liga

            # Eliminar jugadores sin nombre o sin salario (ej: contratos inactivos)
            df = df[df['Player'].str.strip() != '']
            df = df[df['Net_Total_PY_USD'] > 0]

            acumulado.append(df)
            print(f"✅ {os.path.basename(archivo)}: {len(df)} jugadores")

        except Exception as e:
            print(f"⚠️ Fallo en {os.path.basename(archivo)}: {e}")


✅ capology_laliga_22_23_raw.csv: 494 jugadores
✅ capology_laliga_23_24_raw.csv: 505 jugadores
✅ capology_laliga_24_25_raw.csv: 494 jugadores
✅ capology_premier_22_23_raw.csv: 534 jugadores
✅ capology_premier_23_24_raw.csv: 539 jugadores
✅ capology_premier_24_25_raw.csv: 677 jugadores
✅ capology_bundesliga_22_23_raw.csv: 519 jugadores
✅ capology_bundesliga_23_24_raw.csv: 504 jugadores
✅ capology_bundesliga_24_25_raw.csv: 513 jugadores
✅ capology_seriea_22_23_raw.csv: 567 jugadores
✅ capology_seriea_23_24_raw.csv: 556 jugadores
✅ capology_seriea_24_25_raw.csv: 578 jugadores
✅ capology_ligue_22_23_raw.csv: 535 jugadores
✅ capology_ligue_23_24_raw.csv: 469 jugadores
✅ capology_ligue_24_25_raw.csv: 474 jugadores


In [7]:
# ── UNIÓN Y EXPORTACIÓN ──────────────────────────────────────────────────────
if acumulado:
    df_master_salarios = pd.concat(acumulado, ignore_index=True)

    # Orden de columnas: jugador → salarios → metadata
    cols_orden = ['Player', 'Position', 'Age', 'Country', 'Club',
                  'Net_Fixed_PW_USD', 'Net_Fixed_PY_USD', 'Net_Bonus_PY_USD',
                  'Net_Total_PY_USD', 'Adj_Net_Total_PY_2026', 'Temporada', 'Liga']
    df_master_salarios = df_master_salarios[cols_orden]

    ruta_final = r'C:\Users\PAOLA\Desktop\Proyectos personales\Rendimiento_vs_salario\Dashboard_Data\Master_Salarios.csv'
    df_master_salarios.to_csv(ruta_final, index=False, encoding='utf-8-sig')

    print(f"\n✅ Guardado: {len(df_master_salarios)} registros")
    print(f"\nPor liga:\n{df_master_salarios['Liga'].value_counts().to_string()}")
    print(f"\nPor temporada:\n{df_master_salarios['Temporada'].value_counts().to_string()}")
    display(df_master_salarios.head())
else:
    print("🛑 No se procesó ningún archivo.")


✅ Guardado: 7958 registros

Por liga:
Liga
Premier       1750
Serie_a       1701
Bundesliga    1536
La_liga       1493
Ligue         1478

Por temporada:
Temporada
2024-2025    2736
2022-2023    2649
2023-2024    2573


,Player,Position,Age,Country,Club,Net_Fixed_PW_USD,Net_Fixed_PY_USD,Net_Bonus_PY_USD,Net_Total_PY_USD,Adj_Net_Total_PY_2026,Temporada,Liga
0,Eden Hazard,F,32,Belgium,Real Madrid,312186.0,16233675.0,0.0,16233675.0,17130001.0,2022-2023,La_liga
1,Vinicius Junior,F,22,Brazil,Real Madrid,249749.0,12986940.0,6493470.0,19480410.0,20556001.0,2022-2023,La_liga
2,Toni Kroos,M,33,Germany,Real Madrid,243505.0,12662266.0,0.0,12662266.0,13361401.0,2022-2023,La_liga
3,Sergio Busquets,M,34,Spain,Barcelona,239759.0,12467462.0,0.0,12467462.0,13155841.0,2022-2023,La_liga
4,Karim Benzema,F,35,France,Real Madrid,239759.0,12467462.0,0.0,12467462.0,13155841.0,2022-2023,La_liga
